# KoE5 Embedding + Retrieval Evaluation (v2_p2)

파일명 : `embedding_retrieval_eval_v2_p2.ipynb`

이 노트북은 **P2 250개 corpus(v2_p2)** 를 기준으로 KoE5 임베딩을 만들고, Chroma/BM25/RRF/Reranker 조합의 retrieval 성능을 비교하기 위한 실험 노트북입니다. 최종 generation은 여기서 하지 않고, retrieval 품질이 확인된 뒤 별도 LCEL RAG 단계에서 연결합니다.

## 전체 실행 흐름

```text
embedding_retrieval_eval_v2_p2.ipynb
├─ 1. 환경 확인
│  ├─ Python / package version 확인
│  ├─ CUDA 사용 가능 여부 확인
│  └─ GPU 라벨 기록: L4 / RTX4060 / CPU
│
├─ 2. 입력 데이터 로드
│  ├─ outputs/parsing_p2_250/chunks_v2.jsonl
│  │  └─ 기본 corpus: corpus_name=p2_250, corpus_version=v2_p2
│  ├─ outputs/parsing_p2_250/chunks_v1.jsonl
│  │  └─ R0 baseline이 있을 때만 사용
│  └─ data/eval/eval_batch_01.csv ~ eval_batch_25.csv
│     └─ 공식 평가 범위: 500문항
│
├─ 3. 임베딩 생성
│  ├─ 문서 chunk: passage: {chunk_text}
│  ├─ 평가 질문: query: {question}
│  ├─ embedding_model: nlpai-lab/KoE5
│  └─ outputs/retrieval_eval/embedding_cache/ 저장
│
├─ 4. 인덱스 구축
│  ├─ Chroma collection 생성
│  ├─ embed_enabled=True chunk만 적재
│  ├─ toc chunk는 보존하지만 기본 임베딩에서는 제외
│  └─ BM25 sparse index 생성
│
├─ 5. Retrieval 실험 실행
│  ├─ R0: v1 clean text + dense top5
│  ├─ R1: v2_p2 + dense top5
│  ├─ R2: v2_p2 + dense top10
│  ├─ R3: v2_p2 + MMR top5
│  ├─ R4: dense top30 → reranker top5
│  ├─ R5: dense top30 + BM25 top30 → RRF top5
│  └─ R6: dense + BM25 → RRF top30 → reranker top5
│
└─ 6. 평가 및 로그 저장
   ├─ outputs/retrieval_eval/predictions/*.jsonl 저장
   ├─ run_retrieval_eval_extended.py 실행
   ├─ 실험별 *_eval_results_extended.csv 저장
   ├─ 실험별 *_summary_extended.json 저장
   └─ outputs/retrieval_eval/experiment_logs/retrieval_experiments.csv에 append
```

## 생성되는 결과물

```text
outputs/retrieval_eval/
├─ chroma/
│  └─ Chroma persistent DB
├─ embedding_cache/
│  ├─ p2_250_v2_p2_koe5_ids.json
│  ├─ p2_250_v2_p2_koe5_embeddings.npy
│  ├─ p2_250_v1_clean_koe5_ids.json
│  └─ p2_250_v1_clean_koe5_embeddings.npy
├─ predictions/
│  ├─ R1_v2_p2_dense_top5_predictions.jsonl
│  ├─ R2_v2_p2_dense_top10_predictions.jsonl
│  ├─ R3_v2_p2_mmr_top5_predictions.jsonl
│  ├─ R4_v2_p2_dense_rerank_top5_predictions.jsonl
│  ├─ R5_v2_p2_hybrid_rrf_top5_predictions.jsonl
│  └─ R6_v2_p2_hybrid_rrf_rerank_top5_predictions.jsonl
├─ experiment_logs/
│  └─ retrieval_experiments.csv  # 실험 결과 누적 로그
├─ R1_v2_p2_dense_top5_eval_results_extended.csv
├─ R1_v2_p2_dense_top5_summary_extended.json
└─ ...
```

## R0~R6 실험 조건

| ID | 방식 | 특징 | 목적 |
|---|---|---|---|
| R0 | v1 clean text + KoE5 dense top5 | v1 파일이 있을 때만 실행 | clean text baseline 확인 |
| R1 | v2_p2 + KoE5 dense top5 | 가장 단순한 dense retrieval | 구조화 corpus 기본 성능 확인 |
| R2 | v2_p2 + KoE5 dense top10 | top-k를 10으로 확장 | 정답 문서가 뒤쪽에 밀리는지 확인 |
| R3 | v2_p2 + KoE5 MMR top5 | 유사 chunk 중복 완화 | 같은 문서/비슷한 chunk 쏠림 완화 효과 확인 |
| R4 | dense top30 → reranker top5 | 의미 기반 후보를 reranker로 재정렬 | 정밀도 개선 효과 확인 |
| R5 | dense top30 + BM25 top30 → RRF top5 | dense와 sparse 검색 결합 | 기관명/사업명/금액/날짜 exact match 보완 |
| R6 | dense + BM25 → RRF top30 → reranker top5 | hybrid 후보를 reranker로 최종 정렬 | 최종 retrieval 후보 |

## 평가 기준

```text
전체 대표 지표: nDCG@5
single-doc 질문: MRR@5
multi-doc 질문: nDCG@5 + Doc Recall@5
운영 관점: retrieval_ms_p95, rerank_ms_avg, total_retrieval_ms_avg
```

## 빠른 실행 팁

```python
EVAL_SAMPLE_SIZE = 50  # smoke test
EVAL_SAMPLE_SIZE = 0   # 공식 범위 eval_batch_01~25 전체 500문항
```


## GitHub Push Note

GitHub push 전에는 이 노트북의 output을 clear합니다. 실행 결과, prediction JSONL, Chroma DB, embedding cache, 원문/청크 산출물은 Drive 또는 로컬 `outputs/`에만 보관합니다.


## Pilot corpus coverage gate

현재 기본 corpus는 **전체 690개 중 250개만 먼저 구조화한 pilot set** 입니다. 그래서 검색 결과가 없거나 신호가 약한 질문은 실제로 문서에 없는 것이 아니라, 아직 corpus에 포함되지 않았을 가능성이 있습니다.

이 노트북은 generation을 수행하지 않지만, 이후 LCEL/generation 단계에서 그대로 사용할 수 있도록 prediction JSONL에 다음 값을 함께 남깁니다.

```text
retrieval_status
├─ sufficient_context: 검색 근거가 있음
├─ weak_signal: 검색 결과는 있으나 점수가 낮아 확인 필요
└─ insufficient_context: 검색 근거가 없음

coverage_reason
├─ retrieval_signal_passed
├─ top1_score_below_threshold
└─ no_retrieved_context
```

Generation 단계에서는 `retrieval_status == "insufficient_context"`이거나 팀에서 정한 cutoff보다 약하면 답변을 생성하지 않고 **"제공된 문서에서 확인되지 않습니다."** 로 처리합니다. 이 정책은 hallucination 방지용이며, retrieval 성능평가 자체는 기존처럼 모든 query에 대해 수행합니다.



In [2]:
# ===== Install dependencies in the active kernel =====
# VS Code connected to Colab must install packages inside the Colab runtime.
# This cell uses requirements.txt stored in Google Drive.

from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT_FOR_INSTALL = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project")
REQUIREMENTS_PATH = PROJECT_ROOT_FOR_INSTALL / "requirements.txt"

if not REQUIREMENTS_PATH.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Google Drive mount skipped:", exc)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(f"requirements.txt not found: {REQUIREMENTS_PATH}")

REQUIRED_IMPORTS = {
    "chromadb": "chromadb",
    "rank-bm25": "rank_bm25",
    "sentence-transformers": "sentence_transformers",
    "transformers": "transformers",
    "openpyxl": "openpyxl",
}
missing = [name for name, module in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]

if missing:
    print("missing packages:", missing)
    print("installing:", REQUIREMENTS_PATH)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)])
    print("dependency install complete. If imports still fail, restart the Colab runtime once and rerun from the top.")
else:
    print("required packages already installed")


Mounted at /content/drive
missing packages: ['chromadb', 'rank-bm25']
installing: /content/drive/MyDrive/DB_RAG_Codeit_Project/requirements.txt
dependency install complete. If imports still fail, restart the Colab runtime once and rerun from the top.


In [3]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

python: 3.12.13
torch: 2.10.0+cu128
cuda_available: True
gpu: NVIDIA L4


In [40]:
# ===== User parameters =====
# Large input/output files are intentionally not committed. Place chunks/eval CSV on Drive, local disk, or VM before running.

# 0 means official eval set: eval_batch_01~25. Use 30, 50, 100 for a quick smoke test.
EVAL_SAMPLE_SIZE = 0
EVAL_SAMPLE_SEED = 42

CORPUS_NAME = "p2_250"
CORPUS_VERSION = "v2_p2"
CORPUS_COVERAGE = "pilot_250_of_690"
UNVERIFIED_ANSWER = "제공된 문서에서 확인되지 않습니다."
ENABLE_COVERAGE_GATE = True
# Retrieval score scale differs by method. Tune this after first R0~R6 run.
MIN_TOP1_SCORE_FOR_WEAK_SIGNAL = 0.20
EMBEDDING_MODEL = "nlpai-lab/KoE5"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBEDDING_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 32

FINAL_TOP_K = 5
DENSE_TOP_K = 30
BM25_TOP_K = 30
RRF_K = 60
MMR_FETCH_K = 30
MMR_LAMBDA = 0.5

RUN_RERANKER_EXPERIMENTS = True
RUN_R0_IF_V1_EXISTS = True

# If PROJECT_ROOT auto-detection fails, set it manually.
# Google Drive / Colab default:
PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/DB_RAG_Codeit_Project"
# GCV VM example. Uncomment on GCV and comment out the Google Drive line above.
# PROJECT_ROOT_OVERRIDE = "/home/USER/DB_RAG_Codeit_Project"


def detect_project_root() -> Path:
    candidates = []
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    candidates.extend([
        Path.cwd(),
        Path('/content/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project'),
        Path(r'C:/Users/yoosy/Desktop/codeit/project_2nd'),
    ])
    for base in candidates:
        if (base / 'notebooks').exists() or (base / 'requirements.txt').exists():
            return base.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = detect_project_root()
P2_DIR = PROJECT_ROOT / 'outputs' / 'parsing_p2_250'
V2_CHUNKS_PATH = P2_DIR / 'chunks_v2.jsonl'
V1_CHUNKS_PATH = P2_DIR / 'chunks_v1.jsonl'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'retrieval_eval'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
CHROMA_DIR = OUTPUT_DIR / 'chroma'
CACHE_DIR = OUTPUT_DIR / 'embedding_cache'
EXTENDED_EVAL_SCRIPT = PROJECT_ROOT / 'notebooks' / 'eval' / 'evaluation' / 'scripts' / 'run_retrieval_eval_extended.py'

for path in [OUTPUT_DIR, PREDICTION_DIR, CHROMA_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('V2_CHUNKS_PATH exists:', V2_CHUNKS_PATH.exists(), V2_CHUNKS_PATH)
print('V1_CHUNKS_PATH exists:', V1_CHUNKS_PATH.exists(), V1_CHUNKS_PATH)
print('EVAL_DIR exists:', EVAL_DIR.exists(), EVAL_DIR)
print('EXTENDED_EVAL_SCRIPT exists:', EXTENDED_EVAL_SCRIPT.exists(), EXTENDED_EVAL_SCRIPT)


PROJECT_ROOT: /content/drive/MyDrive/DB_RAG_Codeit_Project
V2_CHUNKS_PATH exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p2_250/chunks_v2.jsonl
V1_CHUNKS_PATH exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p2_250/chunks_v1.jsonl
EVAL_DIR exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/data/eval
EXTENDED_EVAL_SCRIPT exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/eval/evaluation/scripts/run_retrieval_eval_extended.py


In [5]:
def gpu_label() -> str:
    if not torch.cuda.is_available():
        return 'CPU'
    name = torch.cuda.get_device_name(0)
    if 'L4' in name.upper():
        return 'L4'
    if '4060' in name:
        return 'RTX4060'
    return 'CUDA'

EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBEDDING_DEVICE_LABEL = gpu_label()
print('EMBEDDING_DEVICE:', EMBEDDING_DEVICE)
print('EMBEDDING_DEVICE_LABEL:', EMBEDDING_DEVICE_LABEL)

EMBEDDING_DEVICE: cuda
EMBEDDING_DEVICE_LABEL: L4


## 1. Load P2 chunks and eval questions


In [41]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_chunks(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.DataFrame(read_jsonl(path))
    if 'content' not in df.columns:
        raise ValueError('chunks JSONL must include content column')
    if 'embed_enabled' not in df.columns:
        df['embed_enabled'] = True
    if 'chunk_type' not in df.columns:
        df['chunk_type'] = 'text'
    df = df.copy()
    df['content'] = df['content'].fillna('').astype(str)
    return df

def select_embedding_chunks(df: pd.DataFrame) -> pd.DataFrame:
    selected = df[df['embed_enabled'].fillna(True).astype(bool)].copy()
    selected = selected[selected['chunk_type'].astype(str).ne('toc')].copy()
    selected = selected[selected['content'].str.strip().ne('')].copy()
    selected = selected.drop_duplicates(subset=['chunk_id']).reset_index(drop=True)
    return selected

chunks_v2_raw = load_chunks(V2_CHUNKS_PATH)
chunks_v2 = select_embedding_chunks(chunks_v2_raw)

summary = pd.DataFrame([
    {'metric': 'raw rows', 'value': len(chunks_v2_raw)},
    {'metric': 'embed_enabled rows', 'value': int(chunks_v2_raw['embed_enabled'].fillna(True).astype(bool).sum())},
    {'metric': 'toc rows', 'value': int(chunks_v2_raw['chunk_type'].astype(str).eq('toc').sum())},
    {'metric': 'selected embedding rows', 'value': len(chunks_v2)},
    {'metric': 'duplicate chunk_id after selection', 'value': int(chunks_v2['chunk_id'].duplicated().sum())},
    {'metric': 'unique docs', 'value': chunks_v2['source_file'].nunique() if 'source_file' in chunks_v2.columns else None},
])
display(summary)
display(chunks_v2[['chunk_id','source_file','chunk_type','content']].head(3))

,metric,value
0,raw rows,41885
1,embed_enabled rows,41637
2,toc rows,248
3,selected embedding rows,41637
4,duplicate chunk_id after selection,0
5,unique docs,250


,chunk_id,source_file,chunk_type,content
0,P001_v2_text_0001_part_001,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n-1-\n제 안 요 청 서\n고려대학교\n차세대 포털·학사 정보시스템 구축 사업
1,P001_v2_text_0002_part_001,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,"[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n※ 본 자료는 제안내용의 설명을 위한 배포자료로, 이외 목적으로 무단복제, 전달 및 사용하는 행..."
2,P001_v2_text_0003_part_001,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n-3-


In [42]:
def parse_list_cell(value) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
        return [str(parsed)]
    except Exception:
        if '|' in text:
            return [x.strip() for x in text.split('|') if x.strip()]
        return [text]

def load_eval_questions(eval_dir: Path, canonical_only: bool = True) -> pd.DataFrame:
    if canonical_only:
        csv_paths = [eval_dir / f'eval_batch_{idx:02d}.csv' for idx in range(1, 26)]
        missing = [str(path) for path in csv_paths if not path.exists()]
        if missing:
            raise FileNotFoundError(f'Missing canonical eval CSV files: {missing}')
    else:
        csv_paths = sorted(eval_dir.glob('eval_batch_*.csv'))
        if not csv_paths:
            raise FileNotFoundError(f'No eval CSV files under {eval_dir}')
    frames = []
    for csv_path in csv_paths:
        frame = pd.read_csv(csv_path)
        frame['source_eval_file'] = csv_path.name
        frames.append(frame)
    df = pd.concat(frames, ignore_index=True)
    required = ['id','type','difficulty','question','ground_truth_answer','ground_truth_docs']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Missing eval columns: {missing}')
    df['ground_truth_doc_list'] = df['ground_truth_docs'].apply(parse_list_cell)
    df['gt_doc_count'] = df['ground_truth_doc_list'].apply(len)
    return df

eval_df_all = load_eval_questions(EVAL_DIR, canonical_only=True)
if EVAL_SAMPLE_SIZE and EVAL_SAMPLE_SIZE > 0:
    eval_df = eval_df_all.sample(n=min(EVAL_SAMPLE_SIZE, len(eval_df_all)), random_state=EVAL_SAMPLE_SEED).sort_values('id').reset_index(drop=True)
else:
    eval_df = eval_df_all.reset_index(drop=True)

print('eval all rows:', len(eval_df_all))
print('eval selected rows:', len(eval_df))
display(eval_df['gt_doc_count'].value_counts().sort_index().rename_axis('gt_doc_count').reset_index(name='rows'))
display(eval_df[['id','type','difficulty','question','gt_doc_count','ground_truth_doc_list']].head(5))

eval all rows: 500
eval selected rows: 500


,gt_doc_count,rows
0,1,297
1,2,164
2,3,32
3,4,7


,id,type,difficulty,question,gt_doc_count,ground_truth_doc_list
0,Q001,A,하,한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?,1,[한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
1,Q002,A,하,고려대학교에서 발주한 '차세대 포털·학사 정보시스템 구축사업'의 예산은 얼마로 배정되어 있나요?,1,[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf]
2,Q003,A,하,코이카(KOICA) 전자조달의 '우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 및 지역의회 연계 개선 PMC 용역' 예산 규모는 어떻게 됩니까?,1,[KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp]
3,Q004,A,중,한국가스공사의 ERP 구축 사업에 도입되는 신규 소프트웨어 내역과 인프라 내역을 모두 나열해 주시겠습니까?,1,[한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
4,Q005,A,중,그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위를 모두 열거해 주십시오.,1,[그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp]


## 2. Build embeddings and Chroma collections


In [8]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=EMBEDDING_DEVICE)
print('loaded embedding model:', EMBEDDING_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

loaded embedding model: nlpai-lab/KoE5


In [9]:
def prefixed_passages(texts: list[str]) -> list[str]:
    return [f'passage: {text}' for text in texts]

def prefixed_queries(texts: list[str]) -> list[str]:
    return [f'query: {text}' for text in texts]

def encode_passages(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    return embedding_model.encode(
        prefixed_passages(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

def encode_queries(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    return embedding_model.encode(
        prefixed_queries(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

def metadata_value(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    if isinstance(value, (list, tuple)):
        return ' > '.join(map(str, value))
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def make_chroma_metadata(row: pd.Series) -> dict[str, str]:
    keys = [
        'doc_id','source_file','norm_name','project_name','issuer','chunk_id','chunk_type',
        'section_path','section_type','final_budget','final_project_duration',
        'final_bid_eligibility_terms','final_submission_documents'
    ]
    return {key: metadata_value(row.get(key, '')) for key in keys if key in row.index}

def embedding_cache_paths(cache_key: str):
    return CACHE_DIR / f'{cache_key}_ids.json', CACHE_DIR / f'{cache_key}_embeddings.npy'

def load_or_create_embeddings(df: pd.DataFrame, cache_key: str) -> np.ndarray:
    ids_path, emb_path = embedding_cache_paths(cache_key)
    ids = df['chunk_id'].astype(str).tolist()
    if ids_path.exists() and emb_path.exists():
        cached_ids = json.loads(ids_path.read_text(encoding='utf-8'))
        if cached_ids == ids:
            print('loading cached embeddings:', emb_path)
            return np.load(emb_path)
    print('encoding passages:', len(ids))
    embeddings = encode_passages(df['content'].astype(str).tolist())
    ids_path.write_text(json.dumps(ids, ensure_ascii=False), encoding='utf-8')
    np.save(emb_path, embeddings)
    return embeddings

def build_or_load_collection(df: pd.DataFrame, collection_name: str, cache_key: str):
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    try:
        client.delete_collection(collection_name)
        print('deleted existing collection:', collection_name)
    except Exception:
        pass
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={'hnsw:space': 'cosine'}
    )
    embeddings = load_or_create_embeddings(df, cache_key)
    ids = df['chunk_id'].astype(str).tolist()
    docs = df['content'].astype(str).tolist()
    metadatas = [make_chroma_metadata(row) for _, row in df.iterrows()]
    batch_size = 1000
    for start in tqdm(range(0, len(ids), batch_size), desc=f'chroma add {collection_name}'):
        end = start + batch_size
        collection.add(
            ids=ids[start:end],
            documents=docs[start:end],
            embeddings=embeddings[start:end].tolist(),
            metadatas=metadatas[start:end],
        )
    print('collection count:', collection.count())
    return collection, embeddings

collection_v2_name = 'rfp_p2_250_v2_p2_koe5'
collection_v2, embeddings_v2 = build_or_load_collection(chunks_v2, collection_v2_name, 'p2_250_v2_p2_koe5')

encoding passages: 41637


Batches:   0%|          | 0/651 [00:00<?, ?it/s]

chroma add rfp_p2_250_v2_p2_koe5:   0%|          | 0/42 [00:00<?, ?it/s]

collection count: 41637


In [10]:
# Optional R0 collection from chunks_v1.jsonl.
collection_v1 = None
chunks_v1 = None
embeddings_v1 = None
collection_v1_name = 'rfp_p2_250_v1_clean_koe5'

if RUN_R0_IF_V1_EXISTS and V1_CHUNKS_PATH.exists():
    chunks_v1_raw = load_chunks(V1_CHUNKS_PATH)
    chunks_v1 = select_embedding_chunks(chunks_v1_raw)
    collection_v1, embeddings_v1 = build_or_load_collection(chunks_v1, collection_v1_name, 'p2_250_v1_clean_koe5')
    print('R0 enabled: chunks_v1 rows =', len(chunks_v1))
else:
    print('R0 skipped: chunks_v1.jsonl not found or disabled')

encoding passages: 31393


Batches:   0%|          | 0/491 [00:00<?, ?it/s]

chroma add rfp_p2_250_v1_clean_koe5:   0%|          | 0/32 [00:00<?, ?it/s]

collection count: 31393
R0 enabled: chunks_v1 rows = 31393


## 3. Retrieval helpers: dense, MMR, BM25, RRF, reranker


In [11]:
def normalize_doc_name(value: str) -> str:
    text = str(value or '').strip().lower()
    text = re.sub(r'\.(hwp|hwpx|pdf|json)$', '', text)
    text = re.sub(r'\s+', '', text)
    return text

def result_from_chroma_item(rank: int, doc: str, metadata: dict, distance: float, embedding=None) -> dict[str, Any]:
    score = 1.0 - float(distance) if distance is not None else math.nan
    return {
        'rank': rank,
        'filename': metadata.get('source_file', ''),
        'doc_id': metadata.get('doc_id', ''),
        'chunk_id': metadata.get('chunk_id', ''),
        'score': score,
        'text': doc,
        'metadata': metadata,
        '_embedding': embedding,
    }

def dense_candidates(collection, question: str, n_results: int) -> tuple[list[dict[str, Any]], np.ndarray, float]:
    t0 = time.perf_counter()
    q_emb = encode_queries([question])[0]
    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=n_results,
        include=['documents','metadatas','distances','embeddings'],
    )
    docs = res.get('documents', [[]])[0]
    metadatas = res.get('metadatas', [[]])[0]
    distances = res.get('distances', [[]])[0]
    embs = res.get('embeddings', [[]])[0]
    items = []
    for idx, (doc, meta, dist) in enumerate(zip(docs, metadatas, distances), start=1):
        emb = embs[idx-1] if len(embs) >= idx else None
        items.append(result_from_chroma_item(idx, doc, meta or {}, dist, emb))
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return items, q_emb, elapsed_ms

def dedupe_by_doc(candidates: list[dict[str, Any]], k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    selected = []
    seen = set()
    for item in candidates:
        doc_key = normalize_doc_name(item.get('filename') or item.get('doc_id'))
        if not doc_key:
            doc_key = str(item.get('doc_id') or item.get('chunk_id'))
        if doc_key in seen:
            continue
        seen.add(doc_key)
        item = dict(item)
        item['rank'] = len(selected) + 1
        selected.append(item)
        if len(selected) >= k:
            break
    return selected

def mmr_select(candidates: list[dict[str, Any]], query_embedding: np.ndarray, k: int, lambda_mult: float = MMR_LAMBDA) -> list[dict[str, Any]]:
    usable = [item for item in candidates if item.get('_embedding') is not None]
    if not usable:
        return candidates[:k]
    cand_embs = np.asarray([item['_embedding'] for item in usable], dtype=np.float32)
    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    query_scores = cand_embs @ query_embedding
    selected_indices = []
    remaining = set(range(len(usable)))
    while remaining and len(selected_indices) < k:
        if not selected_indices:
            chosen = max(remaining, key=lambda idx: query_scores[idx])
        else:
            selected_embs = cand_embs[selected_indices]
            def mmr_score(idx):
                diversity_penalty = float(np.max(selected_embs @ cand_embs[idx]))
                return lambda_mult * float(query_scores[idx]) - (1 - lambda_mult) * diversity_penalty
            chosen = max(remaining, key=mmr_score)
        selected_indices.append(chosen)
        remaining.remove(chosen)
    selected = [dict(usable[idx]) for idx in selected_indices]
    for rank, item in enumerate(selected, start=1):
        item['rank'] = rank
    return selected

def bm25_tokenize(text: str) -> list[str]:
    terms = re.findall(r'[가-힣A-Za-z0-9]+', str(text).lower())
    tokens = []
    for term in terms:
        tokens.append(term)
        if re.search(r'[가-힣]', term) and len(term) >= 2:
            tokens.extend(term[i:i+2] for i in range(len(term)-1))
    return tokens

class BM25Index:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        tokenized = [bm25_tokenize(text) for text in tqdm(self.df['content'].astype(str).tolist(), desc='bm25 tokenize')]
        self.index = BM25Okapi(tokenized)
    def search(self, question: str, top_k: int) -> list[dict[str, Any]]:
        scores = self.index.get_scores(bm25_tokenize(question))
        order = np.argsort(scores)[::-1][:top_k]
        items = []
        for rank, idx in enumerate(order, start=1):
            row = self.df.iloc[int(idx)]
            items.append({
                'rank': rank,
                'filename': row.get('source_file', ''),
                'doc_id': row.get('doc_id', ''),
                'chunk_id': row.get('chunk_id', ''),
                'score': float(scores[idx]),
                'text': row.get('content', ''),
                'metadata': make_chroma_metadata(row),
            })
        return items

def rrf_fuse(result_lists: list[list[dict[str, Any]]], rrf_k: int = RRF_K, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    by_chunk = {}
    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            chunk_id = item.get('chunk_id')
            if not chunk_id:
                continue
            if chunk_id not in by_chunk:
                by_chunk[chunk_id] = dict(item)
                by_chunk[chunk_id]['rrf_score'] = 0.0
            by_chunk[chunk_id]['rrf_score'] += 1.0 / (rrf_k + rank)
    fused = sorted(by_chunk.values(), key=lambda item: item.get('rrf_score', 0.0), reverse=True)[:top_k]
    for rank, item in enumerate(fused, start=1):
        item['rank'] = rank
        item['score'] = float(item.get('rrf_score', item.get('score', 0.0)))
    return fused

def clean_contexts_for_json(items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    clean = []
    for rank, item in enumerate(items, start=1):
        clean.append({
            'rank': rank,
            'filename': item.get('filename', ''),
            'doc_id': item.get('doc_id', ''),
            'chunk_id': item.get('chunk_id', ''),
            'score': float(item.get('score', 0.0)) if item.get('score') is not None else None,
            'text': item.get('text', ''),
            'metadata': item.get('metadata', {}),
        })
    return clean

def context_signal_summary(contexts: list[dict[str, Any]]) -> dict[str, Any]:
    clean_contexts = clean_contexts_for_json(contexts)
    unique_docs = {
        normalize_doc_name(item.get('filename') or item.get('doc_id'))
        for item in clean_contexts
        if item.get('filename') or item.get('doc_id')
    }
    top1_score = None
    if clean_contexts:
        top1_score = clean_contexts[0].get('score')
    return {
        'top1_score': top1_score,
        'top5_doc_count': len(unique_docs),
    }

def infer_retrieval_status(signal: dict[str, Any]) -> tuple[str, str]:
    if signal.get('top5_doc_count', 0) == 0:
        return 'insufficient_context', 'no_retrieved_context'
    top1_score = signal.get('top1_score')
    if ENABLE_COVERAGE_GATE and top1_score is not None and top1_score < MIN_TOP1_SCORE_FOR_WEAK_SIGNAL:
        return 'weak_signal', 'top1_score_below_threshold'
    return 'sufficient_context', 'retrieval_signal_passed'

bm25_v2 = BM25Index(chunks_v2)
bm25_v1 = BM25Index(chunks_v1) if chunks_v1 is not None else None


bm25 tokenize:   0%|          | 0/41637 [00:00<?, ?it/s]

bm25 tokenize:   0%|          | 0/31393 [00:00<?, ?it/s]

In [12]:
reranker = None
if RUN_RERANKER_EXPERIMENTS:
    reranker = CrossEncoder(RERANKER_MODEL, device=EMBEDDING_DEVICE)
    print('loaded reranker:', RERANKER_MODEL)
else:
    print('reranker disabled')

def rerank(question: str, candidates: list[dict[str, Any]], top_k: int = FINAL_TOP_K) -> tuple[list[dict[str, Any]], float]:
    if reranker is None:
        return candidates[:top_k], 0.0
    if not candidates:
        return [], 0.0
    t0 = time.perf_counter()
    pairs = [(question, item.get('text', '')) for item in candidates]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    reranked = []
    for item, score in zip(candidates, scores):
        updated = dict(item)
        updated['score'] = float(score)
        updated['rerank_score'] = float(score)
        reranked.append(updated)
    reranked = sorted(reranked, key=lambda item: item.get('rerank_score', 0.0), reverse=True)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    for rank, item in enumerate(reranked[:top_k], start=1):
        item['rank'] = rank
    return reranked[:top_k], elapsed_ms

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

loaded reranker: BAAI/bge-reranker-v2-m3


## 4. Define R0~R6 experiments


In [43]:
@dataclass
class RetrievalExperiment:
    experiment_id: str
    corpus_name: str
    corpus_version: str
    retrieval_method: str
    collection: Any
    chunks_df: pd.DataFrame
    bm25: Any = None
    collection_name: str = ''
    dense_top_k: int = DENSE_TOP_K
    bm25_top_k: int = 0
    fetch_k: int = FINAL_TOP_K
    rrf_k: int = RRF_K
    reranker_model: str = ''
    rerank_top_k: int = 0
    final_top_k: int = FINAL_TOP_K
    use_mmr: bool = False
    use_hybrid: bool = False
    use_reranker: bool = False

def build_experiments() -> list[RetrievalExperiment]:
    experiments = []
    if collection_v1 is not None and chunks_v1 is not None:
        experiments.append(RetrievalExperiment(
            experiment_id='R0_v1_dense_top5',
            corpus_name='p2_250',
            corpus_version='v1_clean_text',
            retrieval_method='dense',
            collection=collection_v1,
            chunks_df=chunks_v1,
            bm25=bm25_v1,
            collection_name=collection_v1_name,
            dense_top_k=FINAL_TOP_K,
            fetch_k=FINAL_TOP_K,
        ))
    experiments.extend([
        RetrievalExperiment('R1_v2_p2_dense_top5', CORPUS_NAME, CORPUS_VERSION, 'dense', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=FINAL_TOP_K, fetch_k=FINAL_TOP_K),
        RetrievalExperiment('R2_v2_p2_dense_top10', CORPUS_NAME, CORPUS_VERSION, 'dense', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=10, fetch_k=10, final_top_k=10),
        RetrievalExperiment('R3_v2_p2_mmr_top5', CORPUS_NAME, CORPUS_VERSION, 'mmr', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=MMR_FETCH_K, fetch_k=MMR_FETCH_K, use_mmr=True),
        RetrievalExperiment('R4_v2_p2_dense_rerank_top5', CORPUS_NAME, CORPUS_VERSION, 'dense_rerank', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=DENSE_TOP_K, fetch_k=DENSE_TOP_K, use_reranker=True, reranker_model=RERANKER_MODEL, rerank_top_k=FINAL_TOP_K),
        RetrievalExperiment('R5_v2_p2_hybrid_rrf_top5', CORPUS_NAME, CORPUS_VERSION, 'hybrid_rrf', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=DENSE_TOP_K, bm25_top_k=BM25_TOP_K, fetch_k=DENSE_TOP_K, use_hybrid=True),
        RetrievalExperiment('R6_v2_p2_hybrid_rrf_rerank_top5', CORPUS_NAME, CORPUS_VERSION, 'hybrid_rrf_rerank', collection_v2, chunks_v2, bm25_v2, collection_v2_name, dense_top_k=DENSE_TOP_K, bm25_top_k=BM25_TOP_K, fetch_k=DENSE_TOP_K, use_hybrid=True, use_reranker=True, reranker_model=RERANKER_MODEL, rerank_top_k=FINAL_TOP_K),
    ])
    if not RUN_RERANKER_EXPERIMENTS:
        experiments = [exp for exp in experiments if not exp.use_reranker]
    return experiments

experiments = build_experiments()
pd.DataFrame([exp.__dict__ | {'collection': exp.collection_name, 'chunks_df': len(exp.chunks_df), 'bm25': exp.bm25 is not None} for exp in experiments])

,experiment_id,corpus_name,corpus_version,retrieval_method,collection,chunks_df,bm25,collection_name,dense_top_k,bm25_top_k,fetch_k,rrf_k,reranker_model,rerank_top_k,final_top_k,use_mmr,use_hybrid,use_reranker
0,R0_v1_dense_top5,p2_250,v1_clean_text,dense,rfp_p2_250_v1_clean_koe5,31393,True,rfp_p2_250_v1_clean_koe5,5,0,5,60,,0,5,False,False,False
1,R1_v2_p2_dense_top5,p2_250,v2_p2,dense,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,5,0,5,60,,0,5,False,False,False
2,R2_v2_p2_dense_top10,p2_250,v2_p2,dense,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,10,0,10,60,,0,10,False,False,False
3,R3_v2_p2_mmr_top5,p2_250,v2_p2,mmr,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,30,0,30,60,,0,5,True,False,False
4,R4_v2_p2_dense_rerank_top5,p2_250,v2_p2,dense_rerank,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,30,0,30,60,BAAI/bge-reranker-v2-m3,5,5,False,False,True
5,R5_v2_p2_hybrid_rrf_top5,p2_250,v2_p2,hybrid_rrf,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,30,30,30,60,,0,5,False,True,False
6,R6_v2_p2_hybrid_rrf_rerank_top5,p2_250,v2_p2,hybrid_rrf_rerank,rfp_p2_250_v2_p2_koe5,41637,True,rfp_p2_250_v2_p2_koe5,30,30,30,60,BAAI/bge-reranker-v2-m3,5,5,False,True,True


## 5. Run retrieval and write predictions JSONL


In [44]:
def retrieve_one(question: str, exp: RetrievalExperiment) -> tuple[list[dict[str, Any]], float, float, float]:
    retrieval_start = time.perf_counter()
    rerank_ms = 0.0

    dense_items, q_emb, dense_ms = dense_candidates(exp.collection, question, n_results=exp.fetch_k)

    if exp.use_mmr:
        candidates = mmr_select(dense_items, q_emb, k=exp.final_top_k, lambda_mult=MMR_LAMBDA)
    elif exp.use_hybrid:
        bm25_items = exp.bm25.search(question, top_k=exp.bm25_top_k) if exp.bm25 is not None else []
        rrf_top_k = exp.fetch_k if exp.use_reranker else exp.final_top_k
        candidates = rrf_fuse([dense_items, bm25_items], rrf_k=exp.rrf_k, top_k=rrf_top_k)
    else:
        candidates = dense_items[:exp.fetch_k]

    if exp.use_reranker:
        candidates, rerank_ms = rerank(question, candidates, top_k=exp.final_top_k)

    final_contexts = dedupe_by_doc(candidates, k=exp.final_top_k)
    total_ms = (time.perf_counter() - retrieval_start) * 1000
    retrieval_ms = total_ms - rerank_ms
    return final_contexts, retrieval_ms, rerank_ms, total_ms

def retriever_config(exp: RetrievalExperiment) -> dict[str, Any]:
    return {
        'experiment_id': exp.experiment_id,
        'corpus_name': exp.corpus_name,
        'corpus_version': exp.corpus_version,
        'corpus_coverage': CORPUS_COVERAGE,
        'embedding_model': EMBEDDING_MODEL,
        'embedding_device': EMBEDDING_DEVICE_LABEL,
        'vector_db': 'chroma',
        'collection_name': exp.collection_name,
        'retrieval_method': exp.retrieval_method,
        'retriever_type': exp.retrieval_method,
        'dense_top_k': exp.dense_top_k,
        'bm25_top_k': exp.bm25_top_k,
        'fetch_k': exp.fetch_k,
        'rrf_k': exp.rrf_k if exp.use_hybrid else '',
        'reranker': bool(exp.use_reranker),
        'reranker_model': exp.reranker_model,
        'rerank_top_k': exp.rerank_top_k,
        'final_top_k': exp.final_top_k,
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
        'toc_policy': 'excluded_from_embedding',
        'metadata_boost': False,
        'doc_dedup': True,
        'coverage_gate_enabled': ENABLE_COVERAGE_GATE,
        'min_top1_score_for_weak_signal': MIN_TOP1_SCORE_FOR_WEAK_SIGNAL,
        'unverified_answer': UNVERIFIED_ANSWER,
        'eval_sample_size': EVAL_SAMPLE_SIZE,
        'eval_sample_seed': EVAL_SAMPLE_SEED,
    }

def write_predictions(exp: RetrievalExperiment, eval_df: pd.DataFrame) -> Path:
    pred_path = PREDICTION_DIR / f'{exp.experiment_id}_predictions.jsonl'
    config = retriever_config(exp)
    with pred_path.open('w', encoding='utf-8') as f:
        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=exp.experiment_id):
            contexts, retrieval_ms, rerank_ms, total_ms = retrieve_one(row['question'], exp)
            signal = context_signal_summary(contexts)
            retrieval_status, coverage_reason = infer_retrieval_status(signal)
            record = {
                'id': row['id'],
                'source_eval_file': row.get('source_eval_file', ''),
                'question': row['question'],
                'answer': '',
                'retrieved_contexts': clean_contexts_for_json(contexts),
                'corpus_coverage': CORPUS_COVERAGE,
                'retrieval_status': retrieval_status,
                'coverage_reason': coverage_reason,
                'top1_score': signal['top1_score'],
                'top5_doc_count': signal['top5_doc_count'],
                'latency_ms': total_ms,
                'retrieval_ms': retrieval_ms,
                'rerank_ms': rerank_ms,
                'model_name': 'retrieval_only',
                'embedding_model': EMBEDDING_MODEL,
                'retriever_config': config,
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    return pred_path

prediction_paths = []
for exp in experiments:
    path = write_predictions(exp, eval_df)
    prediction_paths.append((exp, path))
    print('saved:', path)


R0_v1_dense_top5:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R0_v1_dense_top5_predictions.jsonl


R1_v2_p2_dense_top5:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R1_v2_p2_dense_top5_predictions.jsonl


R2_v2_p2_dense_top10:   0%|          | 0/500 [00:00<?, ?it/s]

...

saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R4_v2_p2_dense_rerank_top5_predictions.jsonl


R5_v2_p2_hybrid_rrf_top5:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R5_v2_p2_hybrid_rrf_top5_predictions.jsonl


R6_v2_p2_hybrid_rrf_rerank_top5:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R6_v2_p2_hybrid_rrf_rerank_top5_predictions.jsonl


## 6. Run extended evaluation and append cumulative CSV


In [45]:
def run_extended_eval(exp: RetrievalExperiment, pred_path: Path):
    if not EXTENDED_EVAL_SCRIPT.exists():
        raise FileNotFoundError(EXTENDED_EVAL_SCRIPT)
    cmd = [
        sys.executable, str(EXTENDED_EVAL_SCRIPT),
        '--project-root', str(PROJECT_ROOT),
        '--eval-dir', str(EVAL_DIR),
        '--predictions', str(pred_path),
        '--output-dir', str(OUTPUT_DIR),
        '--prediction-ids-only',
        '--top-k', str(exp.final_top_k),
        '--experiment-id', exp.experiment_id,
        '--experiment-name', exp.retrieval_method,
        '--eval-scope', 'sample' if EVAL_SAMPLE_SIZE else 'canonical_01_25',
        '--notes', f'corpus={exp.corpus_name}/{exp.corpus_version}; eval_scope=canonical_01_25; sample_size={EVAL_SAMPLE_SIZE}',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(' '.join(cmd))
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'extended eval failed: {exp.experiment_id}')

for exp, pred_path in prediction_paths:
    run_extended_eval(exp, pred_path)

/usr/bin/python3 /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/eval/evaluation/scripts/run_retrieval_eval_extended.py --project-root /content/drive/MyDrive/DB_RAG_Codeit_Project --eval-dir /content/drive/MyDrive/DB_RAG_Codeit_Project/data/eval --predictions /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R0_v1_dense_top5_predictions.jsonl --output-dir /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval --prediction-ids-only --top-k 5 --experiment-id R0_v1_dense_top5 --experiment-name dense --eval-scope canonical_01_25 --notes corpus=p2_250/v1_clean_text; eval_scope=canonical_01_25; sample_size=0
eval_results: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/R0_v1_dense_top5_eval_results_extended.csv
summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/R0_v1_dense_top5_summary_extended.json
summary_log: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/experiment_logs/re

In [46]:
summary_log = OUTPUT_DIR / 'experiment_logs' / 'retrieval_experiments.csv'
summary_df = pd.read_csv(summary_log)
cols = [
    'experiment_id','run_datetime','corpus_name','corpus_version','embedding_model','embedding_device',
    'retrieval_method','num_eval_questions','num_single_doc_questions','num_multi_doc_questions',
    'hit_at_5_any','mrr_at_5','ndcg_at_5','doc_recall_at_5',
    'single_doc_mrr_at_5','multi_doc_ndcg_at_5','multi_doc_recall_at_5',
    'retrieval_ms_p95','rerank_ms_avg','total_retrieval_ms_avg'
]
existing_cols = [col for col in cols if col in summary_df.columns]
display(summary_df[existing_cols].tail(len(experiments)).sort_values('ndcg_at_5', ascending=False))

,experiment_id,run_datetime,corpus_name,corpus_version,embedding_model,embedding_device,retrieval_method,num_eval_questions,num_single_doc_questions,num_multi_doc_questions,hit_at_5_any,mrr_at_5,ndcg_at_5,doc_recall_at_5,single_doc_mrr_at_5,multi_doc_ndcg_at_5,multi_doc_recall_at_5,retrieval_ms_p95,rerank_ms_avg,total_retrieval_ms_avg
20,R6_v2_p2_hybrid_rrf_rerank_top5,2026-05-18T07:32:40,p2_250,v2_p2,nlpai-lab/KoE5,L4,hybrid_rrf_rerank,500,297,203,0.980,0.958333,0.885728,0.878333,0.943603,0.792695,0.749589,3027.186365,1622.817918,3263.322611
18,R4_v2_p2_dense_rerank_top5,2026-05-18T07:32:39,p2_250,v2_p2,nlpai-lab/KoE5,L4,dense_rerank,500,297,203,0.962,0.933000,0.865772,0.862167,0.923681,0.771505,0.734401,42.585794,1625.642428,1664.598826
19,R5_v2_p2_hybrid_rrf_top5,2026-05-18T07:32:40,p2_250,v2_p2,nlpai-lab/KoE5,L4,hybrid_rrf,500,297,203,0.972,0.944667,0.862844,0.855167,0.921156,0.765654,0.712233,3052.252200,0.000000,1664.979770
17,R3_v2_p2_mmr_top5,2026-05-18T07:32:38,p2_250,v2_p2,nlpai-lab/KoE5,L4,mmr,500,297,203,0.944,0.883567,0.843726,0.870000,0.880864,0.772820,0.788177,42.405312,0.000000,37.971863
16,R2_v2_p2_dense_top10,2026-05-18T07:32:37,p2_250,v2_p2,nlpai-lab/KoE5,L4,dense,500,297,203,0.964,0.885419,0.837258,0.871500,0.889562,0.740053,0.762315,32.275208,0.000000,30.155941
15,R1_v2_p2_dense_top5,2026-05-18T07:32:36,p2_250,v2_p2,nlpai-lab/KoE5,L4,dense,500,297,203,0.944,0.881900,0.802728,0.810667,0.886364,0.664266,0.637110,30.600035,0.000000,28.254163
14,R0_v1_dense_top5,2026-05-18T07:32:35,p2_250,v1_clean_text,nlpai-lab/KoE5,L4,dense,500,297,203,0.942,0.867367,0.789898,0.805667,0.868911,0.654433,0.634647,30.961473,0.000000,28.692139


## 7. 질문별 검색 결과 눈으로 보기

평가지표 숫자만으로는 어떤 질문에서 어떤 문서를 가져왔는지 판단하기 어렵습니다. 아래 셀은 저장된 `predictions/*.jsonl`을 읽어서 질문, 정답 문서, 실제 검색된 문서와 chunk를 함께 보여줍니다.

- 요약표: 질문별 top-k 문서/점수/검색 상태 확인
- 상세보기: 특정 질문 n개를 골라 검색 근거 chunk 원문 확인
- 재검색하지 않고 저장된 prediction 파일을 읽으므로 빠르게 실행됩니다.


In [47]:
# ===== Qualitative retrieval preview: selected experiment only =====

PREVIEW_EXPERIMENT_ID = None  # None이면 가장 최근 prediction 파일을 사용. 예: 'R1_v2_p2_dense_top5'
PREVIEW_N_QUESTIONS = 10  # 정성 확인 기본값. 출력이 길지 않으면 10으로 변경
PREVIEW_CONTEXT_CHARS = 360
PREVIEW_ONLY_MISSES = False  # True면 정답 문서가 top-k에 없는 질문만 우선 확인
SHOW_PREDICTION_FILES_DF = True
SHOW_SINGLE_EXPERIMENT_DF = False  # 긴 텍스트가 잘리므로 기본은 plain text 출력만 사용


def preview_prediction_files(prediction_dir: Path = PREDICTION_DIR) -> pd.DataFrame:
    rows = []
    for path in sorted(prediction_dir.glob('*_predictions.jsonl')):
        experiment_id = path.name.replace('_predictions.jsonl', '')
        rows.append({
            'experiment_id': experiment_id,
            'path': str(path),
            'modified_at': pd.Timestamp.fromtimestamp(path.stat().st_mtime),
            'size_mb': round(path.stat().st_size / 1024 / 1024, 2),
        })
    return pd.DataFrame(rows).sort_values('modified_at', ascending=False).reset_index(drop=True)


def load_prediction_records(experiment_id: str | None = None, prediction_dir: Path = PREDICTION_DIR) -> tuple[str, Path, list[dict[str, Any]]]:
    files_df = preview_prediction_files(prediction_dir)
    if files_df.empty:
        raise FileNotFoundError(f'No prediction files found under {prediction_dir}')
    if experiment_id is None:
        selected = files_df.iloc[0]
    else:
        matches = files_df[files_df['experiment_id'].eq(experiment_id)]
        if matches.empty:
            available = files_df['experiment_id'].tolist()
            raise ValueError(f'Unknown experiment_id={experiment_id}. Available: {available}')
        selected = matches.iloc[0]
    pred_path = Path(selected['path'])
    records = []
    with pred_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return str(selected['experiment_id']), pred_path, records


def eval_lookup_df(eval_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        'id', 'source_eval_file', 'type', 'difficulty', 'question',
        'ground_truth_answer', 'ground_truth_docs', 'ground_truth_doc_list', 'gt_doc_count'
    ]
    existing = [col for col in cols if col in eval_df.columns]
    return eval_df[existing].copy()


def normalize_doc_for_preview(value: str) -> str:
    return normalize_doc_name(value)


def build_retrieval_preview_df(records: list[dict[str, Any]], eval_df: pd.DataFrame | None = None) -> pd.DataFrame:
    lookup = eval_lookup_df(eval_df) if eval_df is not None else pd.DataFrame()
    lookup_by_pair = {}
    lookup_by_id = {}
    if not lookup.empty:
        for _, row in lookup.iterrows():
            item = row.to_dict()
            lookup_by_pair[(str(item.get('source_eval_file', '')), str(item.get('id', '')))] = item
            lookup_by_id[str(item.get('id', ''))] = item

    rows = []
    for rec in records:
        rec_id = str(rec.get('id', ''))
        source_eval_file = str(rec.get('source_eval_file', ''))
        gt = lookup_by_pair.get((source_eval_file, rec_id), lookup_by_id.get(rec_id, {}))
        gt_docs = gt.get('ground_truth_doc_list', []) or []
        contexts = rec.get('retrieved_contexts', []) or []
        retrieved_docs = [ctx.get('filename') or ctx.get('doc_id', '') for ctx in contexts]
        retrieved_chunks = [ctx.get('chunk_id', '') for ctx in contexts]
        scores = [ctx.get('score') for ctx in contexts]
        gt_norm = {normalize_doc_for_preview(doc) for doc in gt_docs}
        retrieved_norm = {normalize_doc_for_preview(doc) for doc in retrieved_docs}
        hit_any = bool(gt_norm & retrieved_norm) if gt_norm else None
        top1_doc = retrieved_docs[0] if retrieved_docs else ''
        top1_text = contexts[0].get('text', '') if contexts else ''
        rows.append({
            'id': rec_id,
            'source_eval_file': source_eval_file,
            'type': gt.get('type', ''),
            'difficulty': gt.get('difficulty', ''),
            'question': rec.get('question', gt.get('question', '')),
            'gt_doc_count': len(gt_docs),
            'ground_truth_docs': ' | '.join(gt_docs),
            'ground_truth_answer': gt.get('ground_truth_answer', ''),
            'retrieval_status': rec.get('retrieval_status', ''),
            'coverage_reason': rec.get('coverage_reason', ''),
            'hit_any_topk': hit_any,
            'top1_score': rec.get('top1_score', scores[0] if scores else None),
            'retrieved_docs': ' | '.join(retrieved_docs),
            'retrieved_chunks': ' | '.join(retrieved_chunks),
            'top1_doc': top1_doc,
            'top1_chunk_preview': str(top1_text).replace('\n', ' ')[:PREVIEW_CONTEXT_CHARS],
            'latency_ms': rec.get('latency_ms'),
        })
    df = pd.DataFrame(rows)
    if PREVIEW_ONLY_MISSES and 'hit_any_topk' in df.columns:
        df = df[df['hit_any_topk'].eq(False)]
    return df


def print_single_experiment_preview(df: pd.DataFrame, n: int = PREVIEW_N_QUESTIONS, context_chars: int = PREVIEW_CONTEXT_CHARS):
    for row_index, row in df.head(n).iterrows():
        print('=' * 120)
        print(f"[{row_index}] {row.get('source_eval_file', '')} / {row.get('id', '')}")
        print('질문:', row.get('question', ''))
        print('정답 문서:', row.get('ground_truth_docs', '') or '(not available)')
        print(f"검색 상태: {row.get('retrieval_status', '')} | hit={row.get('hit_any_topk')} | top1_score={row.get('top1_score')}")
        print('가져온 문서:')
        for doc in str(row.get('retrieved_docs', '')).split(' | '):
            if doc:
                print('  -', doc)
        print('Top1 근거 미리보기:')
        print(str(row.get('top1_chunk_preview', ''))[:context_chars])
        print()


prediction_files_df = preview_prediction_files()
print('prediction files:', len(prediction_files_df))
if SHOW_PREDICTION_FILES_DF:
    display(prediction_files_df)

preview_experiment_id, preview_pred_path, preview_records = load_prediction_records(PREVIEW_EXPERIMENT_ID)
preview_df = build_retrieval_preview_df(preview_records, eval_df)
print('selected experiment:', preview_experiment_id)
print('prediction path:', preview_pred_path)
print('preview rows:', len(preview_df))

if SHOW_SINGLE_EXPERIMENT_DF:
    preview_cols = [
        'id', 'source_eval_file', 'type', 'difficulty', 'question',
        'ground_truth_docs', 'retrieval_status', 'hit_any_topk', 'top1_score',
        'retrieved_docs', 'retrieved_chunks', 'top1_chunk_preview', 'latency_ms'
    ]
    preview_cols = [col for col in preview_cols if col in preview_df.columns]
    display(preview_df[preview_cols].head(PREVIEW_N_QUESTIONS))
else:
    print_single_experiment_preview(preview_df, PREVIEW_N_QUESTIONS, PREVIEW_CONTEXT_CHARS)


prediction files: 7


,experiment_id,path,modified_at,size_mb
0,R6_v2_p2_hybrid_rrf_rerank_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R6_v2_p2_hybrid_rrf_rerank_top5_predictions.jsonl,2026-05-18 07:32:34,4.56
1,R5_v2_p2_hybrid_rrf_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R5_v2_p2_hybrid_rrf_top5_predictions.jsonl,2026-05-18 07:05:22,4.15
2,R4_v2_p2_dense_rerank_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R4_v2_p2_dense_rerank_top5_predictions.jsonl,2026-05-18 06:51:29,4.78
3,R3_v2_p2_mmr_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R3_v2_p2_mmr_top5_predictions.jsonl,2026-05-18 06:37:36,8.01
4,R2_v2_p2_dense_top10,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R2_v2_p2_dense_top10_predictions.jsonl,2026-05-18 06:37:17,8.05
5,R1_v2_p2_dense_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R1_v2_p2_dense_top5_predictions.jsonl,2026-05-18 06:37:01,5.09
6,R0_v1_dense_top5,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R0_v1_dense_top5_predictions.jsonl,2026-05-18 06:36:47,5.75


selected experiment: R6_v2_p2_hybrid_rrf_rerank_top5
prediction path: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval/predictions/R6_v2_p2_hybrid_rrf_rerank_top5_predictions.jsonl
preview rows: 500
[0] eval_batch_01.csv / Q001
질문: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
정답 문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
검색 상태: sufficient_context | hit=True | top1_score=0.8785936236381531
가져온 문서:
  - 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
  - 한국기계연구원_(재공고)기계(연) 차세대 통합정보시스템 구축.hwp
Top1 근거 미리보기:
[문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차세대 통합정보시스템(ERP) 구축 | 발주기관: 한국가스공사 | 섹션: Ⅲ. 일반사항 | 유형: text] Ⅲ. 일반사항 1. 사 업 명 : 차세대 통합정보시스템(ERP) 구축 개요 2. 사업목적 ○ 기술지원 종료(‘27년)에 대비한 ERP 업그레이드 - 제조사(SAP社)의 기술지원 종료 이후 세법 개정, 보안취약점 발생시 원활한 대응이 불가 등 기술지원 종료 대비한 선제적 대응으로 안정적 서비스 제공 ○ 정보시스템 복잡도 해소 및 접근‧활용 간소화로 업무생산성 향상 - ‘09년 도입 이후 종합적인 개선 없이 단편적 수정으로 복잡도 증가 및 

[1] eval_batch_01.csv / Q002
질문: 고려대학교에서 발주한 '차세대 포털·학사 정보시스템 구축사업'의 예산은 얼마로 배정되어 있나요?
정답 문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf

### R0~R6 전체 비교 preview

위 요약표는 선택한 실험 1개를 자세히 보는 용도입니다. 아래 셀은 저장된 모든 `*_predictions.jsonl`을 읽어서 같은 질문에 대해 R0~R6가 각각 어떤 문서를 가져왔는지 비교합니다.


In [48]:
# ===== All-experiment qualitative preview =====

ALL_PREVIEW_N_QUESTIONS = 10  # 기본 10개. 출력이 너무 길면 5로 변경
ALL_PREVIEW_DOC_CHARS = 150
ALL_PREVIEW_CONTEXT_CHARS = 260
ALL_PREVIEW_ONLY_MISSES = False
SHOW_ALL_EXPERIMENT_WIDE_DF = False

# 기본 비교 대상은 R1~R6 6가지 기법입니다. R0는 v1 baseline이므로 필요할 때만 포함합니다.
INCLUDE_R0_BASELINE_IN_PREVIEW = False
ALL_PREVIEW_EXPERIMENT_IDS = [
    'R1_v2_p2_dense_top5',
    'R2_v2_p2_dense_top10',
    'R3_v2_p2_mmr_top5',
    'R4_v2_p2_dense_rerank_top5',
    'R5_v2_p2_hybrid_rrf_top5',
    'R6_v2_p2_hybrid_rrf_rerank_top5',
]
if INCLUDE_R0_BASELINE_IN_PREVIEW:
    ALL_PREVIEW_EXPERIMENT_IDS = ['R0_v1_dense_top5'] + ALL_PREVIEW_EXPERIMENT_IDS


def selected_preview_experiments(prediction_dir: Path = PREDICTION_DIR) -> list[str]:
    files_df = preview_prediction_files(prediction_dir)
    available = set(files_df['experiment_id'].tolist()) if not files_df.empty else set()
    selected = [exp_id for exp_id in ALL_PREVIEW_EXPERIMENT_IDS if exp_id in available]
    missing = [exp_id for exp_id in ALL_PREVIEW_EXPERIMENT_IDS if exp_id not in available]
    print('selected experiments:', selected)
    if missing:
        print('missing prediction files:', missing)
    return selected


def build_all_experiment_preview_df(prediction_dir: Path = PREDICTION_DIR, eval_df: pd.DataFrame | None = None) -> pd.DataFrame:
    selected_experiments = selected_preview_experiments(prediction_dir)
    if not selected_experiments:
        raise FileNotFoundError(f'No selected prediction files found under {prediction_dir}')

    frames = []
    for experiment_id in selected_experiments:
        _, _, records = load_prediction_records(experiment_id, prediction_dir)
        frame = build_retrieval_preview_df(records, eval_df)
        frame.insert(0, 'experiment_id', experiment_id)
        frames.append(frame)
    all_df = pd.concat(frames, ignore_index=True)
    if ALL_PREVIEW_ONLY_MISSES and 'hit_any_topk' in all_df.columns:
        all_df = all_df[all_df['hit_any_topk'].eq(False)]
    return all_df


def print_question_across_experiments(all_df: pd.DataFrame, n_questions: int = ALL_PREVIEW_N_QUESTIONS):
    if all_df.empty:
        print('No preview rows.')
        return

    # 질문 순서는 현재 선택된 preview_df 기준을 우선 사용한다.
    if 'preview_df' in globals() and not preview_df.empty:
        base_questions = preview_df[['source_eval_file', 'id', 'question', 'ground_truth_docs']].head(n_questions)
    else:
        base_questions = all_df[['source_eval_file', 'id', 'question', 'ground_truth_docs']].drop_duplicates().head(n_questions)

    experiment_order = [exp_id for exp_id in ALL_PREVIEW_EXPERIMENT_IDS if exp_id in set(all_df['experiment_id'])]
    for _, qrow in base_questions.iterrows():
        source_eval_file = str(qrow.get('source_eval_file', ''))
        qid = str(qrow.get('id', ''))
        subset = all_df[
            all_df['source_eval_file'].astype(str).eq(source_eval_file)
            & all_df['id'].astype(str).eq(qid)
        ].copy()
        if subset.empty:
            continue
        subset['experiment_id'] = pd.Categorical(subset['experiment_id'], categories=experiment_order, ordered=True)
        subset = subset.sort_values('experiment_id')

        print('=' * 120)
        print(f"[{source_eval_file} / {qid}]")
        print('질문:', qrow.get('question', ''))
        print('정답 문서:', qrow.get('ground_truth_docs', ''))
        print('-' * 120)

        for _, row in subset.iterrows():
            docs = str(row.get('retrieved_docs', '')).replace(' | ', '\n    - ')
            if len(docs) > ALL_PREVIEW_DOC_CHARS:
                docs = docs[:ALL_PREVIEW_DOC_CHARS] + ' ...(생략)'
            context = str(row.get('top1_chunk_preview', ''))
            if len(context) > ALL_PREVIEW_CONTEXT_CHARS:
                context = context[:ALL_PREVIEW_CONTEXT_CHARS] + ' ...(생략)'
            print(f"{row.get('experiment_id')} | hit={row.get('hit_any_topk')} | top1_score={row.get('top1_score')} | status={row.get('retrieval_status')}")
            print('  가져온 문서:')
            print('    - ' + docs)
            print('  top1 근거:', context)
            print()


def make_all_experiment_wide_df(all_df: pd.DataFrame, n_questions: int = ALL_PREVIEW_N_QUESTIONS) -> pd.DataFrame:
    base = all_df[['source_eval_file', 'id', 'question', 'ground_truth_docs']].drop_duplicates().head(n_questions)
    rows = []
    for _, qrow in base.iterrows():
        source_eval_file = str(qrow['source_eval_file'])
        qid = str(qrow['id'])
        item = {
            'source_eval_file': source_eval_file,
            'id': qid,
            'question': qrow['question'],
            'ground_truth_docs': qrow['ground_truth_docs'],
        }
        subset = all_df[
            all_df['source_eval_file'].astype(str).eq(source_eval_file)
            & all_df['id'].astype(str).eq(qid)
        ]
        for _, row in subset.iterrows():
            exp = row['experiment_id']
            item[f'{exp}_hit'] = row.get('hit_any_topk')
            item[f'{exp}_top1'] = row.get('top1_doc')
        rows.append(item)
    return pd.DataFrame(rows)


all_experiment_preview_df = build_all_experiment_preview_df(PREDICTION_DIR, eval_df)
print('all experiment preview rows:', len(all_experiment_preview_df))
print_question_across_experiments(all_experiment_preview_df, ALL_PREVIEW_N_QUESTIONS)

if SHOW_ALL_EXPERIMENT_WIDE_DF:
    all_experiment_wide_df = make_all_experiment_wide_df(all_experiment_preview_df, ALL_PREVIEW_N_QUESTIONS)
    display(all_experiment_wide_df)


selected experiments: ['R1_v2_p2_dense_top5', 'R2_v2_p2_dense_top10', 'R3_v2_p2_mmr_top5', 'R4_v2_p2_dense_rerank_top5', 'R5_v2_p2_hybrid_rrf_top5', 'R6_v2_p2_hybrid_rrf_rerank_top5']
all experiment preview rows: 3000
[eval_batch_01.csv / Q001]
질문: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
정답 문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
------------------------------------------------------------------------------------------------------------------------
R1_v2_p2_dense_top5 | hit=True | top1_score=0.7302561402320862 | status=sufficient_context
  가져온 문서:
    - 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
  top1 근거: [문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차세대 통합정보시스템(ERP) 구축 | 발주기관: 한국가스공사 | 섹션: Ⅲ. 일반사항 | 유형: text] Ⅲ. 일반사항 1. 사 업 명 : 차세대 통합정보시스템(ERP) 구축 개요 2. 사업목적 ○ 기술지원 종료(‘27년)에 대비한 ERP 업그레이드 - 제조사(SAP社)의 기술지원 종료 이후 세법 개정, 보안취약점 발생시 원활한 대응이 불가 등 기술지원 종료 대비한 ...(생략)

R2_v2_p2_dense_top10 | hit=True | top1_score=0.7302561402320862 | status=sufficient_context
  가져온 문서:
    - 한국가스공사_[

In [49]:
# ===== Show multiple questions with full retrieved evidence chunks =====

DETAIL_ROW_INDICES = list(range(5))  # preview_df 기준 행 번호 목록. 예: [0, 1, 2, 3, 4]
DETAIL_CONTEXT_CHARS = 900
DETAIL_MAX_CONTEXTS = 3  # 질문당 보여줄 retrieved chunk 수. 전체 top5를 보려면 5로 변경


def show_retrieval_case(row_index: int, context_chars: int = DETAIL_CONTEXT_CHARS, max_contexts: int = DETAIL_MAX_CONTEXTS):
    if 'preview_records' not in globals() or not preview_records:
        raise RuntimeError('Run the preview summary cell first.')
    if row_index < 0 or row_index >= len(preview_records):
        raise IndexError(f'row_index must be between 0 and {len(preview_records)-1}')

    rec = preview_records[row_index]
    row = preview_df.iloc[row_index].to_dict() if 'preview_df' in globals() and len(preview_df) > row_index else {}
    print('=' * 120)
    print(f"row_index     : {row_index}")
    print(f"experiment_id : {preview_experiment_id}")
    print(f"question_id   : {rec.get('id', '')}")
    print(f"eval_file     : {rec.get('source_eval_file', '')}")
    print(f"status        : {rec.get('retrieval_status', '')} / {rec.get('coverage_reason', '')}")
    print(f"latency_ms    : {rec.get('latency_ms', ''):.2f}" if isinstance(rec.get('latency_ms'), (int, float)) else f"latency_ms    : {rec.get('latency_ms', '')}")
    print('-' * 120)
    print('[QUESTION]')
    print(rec.get('question', ''))
    print('-' * 120)
    print('[GROUND TRUTH DOCS]')
    print(row.get('ground_truth_docs', '') or '(not available)')
    if row.get('ground_truth_answer'):
        print('-' * 120)
        print('[GROUND TRUTH ANSWER]')
        print(row.get('ground_truth_answer'))
    print('=' * 120)

    contexts = rec.get('retrieved_contexts', []) or []
    for ctx in contexts[:max_contexts]:
        meta = ctx.get('metadata', {}) or {}
        text = str(ctx.get('text', '')).strip()
        if len(text) > context_chars:
            text = text[:context_chars] + '\n...(생략)...'
        print(f"\n[RANK {ctx.get('rank')}] score={ctx.get('score')}")
        print(f"filename    : {ctx.get('filename', '')}")
        print(f"doc_id      : {ctx.get('doc_id', '')}")
        print(f"chunk_id    : {ctx.get('chunk_id', '')}")
        print(f"chunk_type  : {meta.get('chunk_type', '')}")
        print(f"section_path: {meta.get('section_path', '')}")
        print('-' * 120)
        print(text)
        print('-' * 120)


def show_retrieval_cases(row_indices=None, context_chars: int = DETAIL_CONTEXT_CHARS, max_contexts: int = DETAIL_MAX_CONTEXTS):
    if row_indices is None:
        row_indices = DETAIL_ROW_INDICES
    if isinstance(row_indices, int):
        row_indices = list(range(row_indices))
    for row_index in row_indices:
        show_retrieval_case(int(row_index), context_chars=context_chars, max_contexts=max_contexts)


show_retrieval_cases(DETAIL_ROW_INDICES, DETAIL_CONTEXT_CHARS, DETAIL_MAX_CONTEXTS)


row_index     : 0
experiment_id : R6_v2_p2_hybrid_rrf_rerank_top5
question_id   : Q001
eval_file     : eval_batch_01.csv
status        : sufficient_context / retrieval_signal_passed
latency_ms    : 2224.73
------------------------------------------------------------------------------------------------------------------------
[QUESTION]
한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
------------------------------------------------------------------------------------------------------------------------
[GROUND TRUTH DOCS]
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
------------------------------------------------------------------------------------------------------------------------
[GROUND TRUTH ANSWER]
해당 사업의 예산 규모는 14,107,009,000원입니다.

[RANK 1] score=0.8785936236381531
filename    : 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
doc_id      : doc_3a265f371a
chunk_id    : P009_v2_text_0004_part_001
chunk_type  : text
section_path: Ⅲ. 일반사항
-----------------------------------------------------------------

In [50]:
# ===== Compare multiple questions across experiments =====

COMPARE_ROW_INDICES = list(range(5))  # preview_df 기준 질문 목록. 예: [0, 1, 2, 3, 4], 출력 가능하면 list(range(10))
COMPARE_CONTEXT_CHARS = 180
COMPARE_TOP_DOC_CHARS = 150
SHOW_COMPARE_DF = False


def _read_prediction_lookup(prediction_dir: Path = PREDICTION_DIR) -> dict[tuple[str, str, str], dict[str, Any]]:
    lookup = {}
    selected_experiments = selected_preview_experiments(prediction_dir)
    files_df = preview_prediction_files(prediction_dir)
    files_df = files_df[files_df['experiment_id'].isin(selected_experiments)]
    for _, file_row in files_df.iterrows():
        exp_id = str(file_row['experiment_id'])
        pred_path = Path(file_row['path'])
        with pred_path.open('r', encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                key = (exp_id, str(rec.get('source_eval_file', '')), str(rec.get('id', '')))
                lookup[key] = rec
    return lookup


def compare_questions_across_experiments(row_indices=None, context_chars: int = COMPARE_CONTEXT_CHARS) -> pd.DataFrame:
    if 'preview_df' not in globals() or preview_df.empty:
        raise RuntimeError('Run the preview summary cell first.')
    if row_indices is None:
        row_indices = COMPARE_ROW_INDICES
    if isinstance(row_indices, int):
        row_indices = list(range(row_indices))

    experiments_sorted = selected_preview_experiments(PREDICTION_DIR)
    pred_lookup = _read_prediction_lookup(PREDICTION_DIR)
    rows = []

    for row_index in row_indices:
        qrow = preview_df.iloc[int(row_index)]
        source_eval_file = str(qrow.get('source_eval_file', ''))
        qid = str(qrow.get('id', ''))
        print('=' * 120)
        print(f"row_index: {row_index} | {source_eval_file} / {qid}")
        print('질문:', qrow.get('question', ''))
        print('정답 문서:', qrow.get('ground_truth_docs', ''))
        print('-' * 120)

        for exp_id in experiments_sorted:
            rec = pred_lookup.get((exp_id, source_eval_file, qid))
            if rec is None:
                continue
            contexts = rec.get('retrieved_contexts', []) or []
            top_docs = [ctx.get('filename') or ctx.get('doc_id', '') for ctx in contexts]
            top_scores = [ctx.get('score') for ctx in contexts]
            top_preview = str(contexts[0].get('text', '')).replace('\n', ' ')[:context_chars] if contexts else ''
            top_docs_text = ' | '.join(top_docs)
            if len(top_docs_text) > COMPARE_TOP_DOC_CHARS:
                top_docs_text = top_docs_text[:COMPARE_TOP_DOC_CHARS] + ' ...(생략)...'
            print(f"{exp_id} | status={rec.get('retrieval_status', '')} | top1_score={rec.get('top1_score', top_scores[0] if top_scores else None)}")
            print('  top_docs:', top_docs_text)
            print('  top1_preview:', top_preview)
            print()
            rows.append({
                'row_index': row_index,
                'experiment_id': exp_id,
                'source_eval_file': source_eval_file,
                'question_id': qid,
                'question': qrow.get('question', ''),
                'ground_truth_docs': qrow.get('ground_truth_docs', ''),
                'retrieval_status': rec.get('retrieval_status', ''),
                'top1_score': rec.get('top1_score', top_scores[0] if top_scores else None),
                'top_docs': ' | '.join(top_docs),
                'top1_preview': top_preview,
                'latency_ms': rec.get('latency_ms'),
            })
    result = pd.DataFrame(rows)
    if SHOW_COMPARE_DF:
        display(result)
    return result


compare_question_result = compare_questions_across_experiments(COMPARE_ROW_INDICES, COMPARE_CONTEXT_CHARS)


selected experiments: ['R1_v2_p2_dense_top5', 'R2_v2_p2_dense_top10', 'R3_v2_p2_mmr_top5', 'R4_v2_p2_dense_rerank_top5', 'R5_v2_p2_hybrid_rrf_top5', 'R6_v2_p2_hybrid_rrf_rerank_top5']
selected experiments: ['R1_v2_p2_dense_top5', 'R2_v2_p2_dense_top10', 'R3_v2_p2_mmr_top5', 'R4_v2_p2_dense_rerank_top5', 'R5_v2_p2_hybrid_rrf_top5', 'R6_v2_p2_hybrid_rrf_rerank_top5']
row_index: 0 | eval_batch_01.csv / Q001
질문: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
정답 문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
------------------------------------------------------------------------------------------------------------------------
R1_v2_p2_dense_top5 | status=sufficient_context | top1_score=0.7302561402320862
  top_docs: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
  top1_preview: [문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차세대 통합정보시스템(ERP) 구축 | 발주기관: 한국가스공사 | 섹션: Ⅲ. 일반사항 | 유형: text] Ⅲ. 일반사항 1. 사 업 명 : 차세대 통합정보시스템(ERP) 구축 개요 2. 사업목적 ○ 기술지원 종료(‘27년)에

R2_v2_p2_dense_top10 | status=sufficient_context | 

## 8. Reading the results

- 전체 대표 지표: `ndcg_at_5`
- 단일 문서 질문 subset: `single_doc_mrr_at_5`
- 다중 문서 질문 subset: `multi_doc_ndcg_at_5`, `multi_doc_recall_at_5`
- 운영 관점 latency: `retrieval_ms_p95`, `rerank_ms_avg`, `total_retrieval_ms_avg`

초기 smoke test는 `EVAL_SAMPLE_SIZE=30~100`으로 실행하고, 최종 비교는 `EVAL_SAMPLE_SIZE=0`으로 공식 범위인 eval_batch_01~25 전체를 실행합니다. eval_batch_26 이후는 Q id가 다시 시작하므로 기본 평가에서 제외합니다.
